# NovaRetail Customer Intelligence Dashboard

**Prepared for:** Sophia Martinez, Director of Customer Intelligence
**Purpose:** Explore customer behavior, revenue patterns, and performance across regions, product categories, and sales channels — to spot growth opportunities, flag early signs of customer decline, and support strategic investment decisions.

This notebook mirrors the interactive HTML dashboard, built with `pandas` + `plotly` for the visuals and `ipywidgets` for live filtering. Run all cells in Jupyter (with a kernel active) to get the same cross-filtering experience — segment, region, channel, category, age, and gender filters all update every chart and table together.

> **Note:** Keep `NR_dataset.xlsx` in the same folder as this notebook — the load cell below reads it from the local directory.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

# Brand palette — matches the HTML dashboard
SEG_COLOR = {
    'Promising': '#1F8A70',
    'Growth':    '#2E6FA7',
    'Stable':    '#C08A2E',
    'Decline':   '#B23A48',
}
SEGMENTS = ['Promising', 'Growth', 'Stable', 'Decline']
NAVY = '#0F1B33'
FONT = 'Inter, Space Grotesk, sans-serif'

## 1. Load & clean the data

The raw file has ~35 inconsistent product-category labels (e.g. *Grocery*, *Groceries*, *Grocery Items*) and one transaction missing its behavioral segment label. Both are cleaned up here.

In [2]:
df = pd.read_excel('NR_dataset.xlsx', sheet_name='data')

# --- Normalize the ~35 raw product categories into 7 clean groups ---
CATEGORY_MAP = {
    'Electronics':'Electronics', 'Gaming':'Electronics',
    'Clothing':'Clothing', 'Fashion':'Clothing', 'Fashion & Apparel':'Clothing',
    'Fashion Accessories':'Clothing', "Children's Clothing":'Clothing', 'Sportswear':'Clothing',
    'Grocery':'Groceries', 'Groceries':'Groceries', 'Grocery Items':'Groceries', 'Food & Beverages':'Groceries',
    'Books':'Books', 'Books & Magazines':'Books',
    'Home Appliances':'Home Products', 'Furniture':'Home Products', 'Furniture & Decor':'Home Products',
    'Home & Garden':'Home Products', 'Home Decor':'Home Products', 'Home Improvement':'Home Products',
    'Gardening Tools':'Home Products', 'Office Supplies':'Home Products',
    'Cosmetics':'Health & Beauty', 'Health & Beauty':'Health & Beauty', 'Beauty & Personal Care':'Health & Beauty',
    'Beauty Products':'Health & Beauty', 'Health & Wellness':'Health & Beauty', 'Health Supplements':'Health & Beauty',
    'Toys':'Sports & Toys', 'Toys & Games':'Sports & Toys', 'Sports Equipment':'Sports & Toys',
    'Sports & Outdoors':'Sports & Toys', 'Sporting Goods':'Sports & Toys', 'Outdoor Equipment':'Sports & Toys',
    'Automotive':'Sports & Toys',
}
df['Category'] = df['ProductCategory'].map(CATEGORY_MAP).fillna('Other')

# --- Fix the one missing segment label (satisfaction score of 1 -> Decline pattern) ---
df['label'] = df['label'].fillna('Decline')

df['TransactionDate'] = pd.to_datetime(df['TransactionDate'])
df = df.rename(columns={'label': 'Segment'})

print(f"{len(df)} transactions | {df['CustomerID'].nunique()} unique customers | "
      f"{df['TransactionDate'].min().date()} to {df['TransactionDate'].max().date()}")
df.head()

100 transactions | 34 unique customers | 2023-01-15 to 2023-02-20


,idx,Segment,CustomerID,TransactionID,TransactionDate,ProductCategory,PurchaseAmount,CustomerAgeGroup,CustomerGender,CustomerRegion,CustomerSatisfaction,RetailChannel,Category
0,0,Promising,12345,TX1001,2023-01-15,Electronics,349.99,25-34,Male,North,4,Online,Electronics
1,1,Growth,12346,TX1002,2023-01-16,Home Appliances,199.99,35-44,Female,West,3,Physical Store,Home Products
2,2,Promising,12347,TX1003,2023-01-17,Clothing,89.99,18-24,Male,South,5,Online,Clothing
3,3,Decline,12348,TX1004,2023-01-18,Groceries,59.99,45-54,Female,East,2,Physical Store,Groceries
4,4,Stable,12349,TX1005,2023-01-19,Books,29.99,25-34,Male,North,4,Online,Books


## 2. KPI summary

In [3]:
def kpi_summary(data):
    total_rev = data['PurchaseAmount'].sum()
    total_tx = len(data)
    unique_customers = data['CustomerID'].nunique()
    avg_order = data['PurchaseAmount'].mean() if total_tx else 0
    avg_csat = data['CustomerSatisfaction'].mean() if total_tx else 0
    decline_rev = data.loc[data['Segment'] == 'Decline', 'PurchaseAmount'].sum()
    decline_pct = (decline_rev / total_rev * 100) if total_rev else 0

    summary = pd.DataFrame({
        'Metric': ['Total Revenue', 'Transactions', 'Unique Customers',
                   'Avg Order Value', 'Avg Satisfaction (1-5)', 'Revenue at Risk (Decline)'],
        'Value': [f'${total_rev:,.0f}', f'{total_tx}', f'{unique_customers}',
                  f'${avg_order:,.2f}', f'{avg_csat:.2f}',
                  f'${decline_rev:,.0f}  ({decline_pct:.1f}% of revenue)']
    })
    return summary.style.hide(axis='index').set_properties(**{'text-align': 'left'})

kpi_summary(df)

Metric,Value
Total Revenue,"$16,969"
Transactions,100
Unique Customers,34
Avg Order Value,$169.69
Avg Satisfaction (1-5),3.71
Revenue at Risk (Decline),"$3,900 (23.0% of revenue)"


## 3. Customer lifecycle — revenue & volume by segment

Each behavioral segment is a stage: **Promising → Growth → Stable → Decline**. Bar height reflects revenue; labels show transaction counts — a quick read on where money and risk concentrate.

In [4]:
def lifecycle_chart(data):
    seg_stats = (data.groupby('Segment')
                      .agg(Revenue=('PurchaseAmount', 'sum'), Transactions=('PurchaseAmount', 'count'))
                      .reindex(SEGMENTS).fillna(0))

    fig = go.Figure(go.Bar(
        x=seg_stats.index, y=seg_stats['Revenue'],
        marker_color=[SEG_COLOR[s] for s in seg_stats.index],
        text=[f"${v:,.0f}<br>{int(c)} tx" for v, c in zip(seg_stats['Revenue'], seg_stats['Transactions'])],
        textposition='outside',
    ))
    fig.update_layout(
        title='Revenue by Lifecycle Stage', font_family=FONT, height=380,
        yaxis_title='Revenue ($)', xaxis_title=None, showlegend=False,
        plot_bgcolor='white', margin=dict(t=60, b=20),
    )
    return fig

lifecycle_chart(df).show()

## 4. Revenue trend over time

In [5]:
def trend_chart(data):
    daily = data.groupby('TransactionDate')['PurchaseAmount'].sum().reset_index()
    fig = px.area(daily, x='TransactionDate', y='PurchaseAmount',
                  labels={'TransactionDate': '', 'PurchaseAmount': 'Revenue ($)'})
    fig.update_traces(line_color='#2E6FA7', fillcolor='rgba(46,111,167,0.12)')
    fig.update_layout(title='Daily Revenue', font_family=FONT, height=350,
                       plot_bgcolor='white', margin=dict(t=60, b=20))
    return fig

trend_chart(df).show()

## 5. Revenue by region, category & channel

In [6]:
def region_category_channel(data):
    fig = make_subplots(
        rows=1, cols=3, specs=[[{'type':'bar'}, {'type':'bar'}, {'type':'domain'}]],
        subplot_titles=('Revenue by Region', 'Top Categories by Revenue', 'Channel Split')
    )

    region_rev = data.groupby('CustomerRegion')['PurchaseAmount'].sum().sort_values(ascending=False)
    fig.add_trace(go.Bar(x=region_rev.index, y=region_rev.values, marker_color='#2E6FA7', showlegend=False), row=1, col=1)

    cat_rev = data.groupby('Category')['PurchaseAmount'].sum().sort_values(ascending=False).head(6)
    fig.add_trace(go.Bar(x=cat_rev.values, y=cat_rev.index, orientation='h', marker_color='#1F8A70', showlegend=False), row=1, col=2)

    chan_rev = data.groupby('RetailChannel')['PurchaseAmount'].sum()
    fig.add_trace(go.Pie(labels=chan_rev.index, values=chan_rev.values,
                         marker_colors=['#2E6FA7', '#C08A2E'], hole=0.55, showlegend=True), row=1, col=3)

    fig.update_layout(font_family=FONT, height=380, plot_bgcolor='white', margin=dict(t=60, b=20))
    return fig

region_category_channel(df).show()

## 6. Satisfaction by segment — early warning signal

In [7]:
def satisfaction_chart(data):
    sat = data.groupby('Segment')['CustomerSatisfaction'].mean().reindex(SEGMENTS)
    fig = go.Figure(go.Bar(x=sat.index, y=sat.values,
                           marker_color=[SEG_COLOR[s] for s in sat.index],
                           text=[f'{v:.2f}' for v in sat.values], textposition='outside'))
    fig.update_layout(title='Avg Satisfaction (1–5) by Segment', font_family=FONT, height=350,
                       yaxis=dict(range=[0, 5.3], title='Avg CSAT'), plot_bgcolor='white',
                       margin=dict(t=60, b=20), showlegend=False)
    return fig

satisfaction_chart(df).show()

## 7. Revenue by age group

In [8]:
AGE_ORDER = ['18-24', '25-34', '35-44', '45-54', '55-64', '55+']

def age_chart(data):
    order = [a for a in AGE_ORDER if a in data['CustomerAgeGroup'].unique()]
    age_rev = data.groupby('CustomerAgeGroup')['PurchaseAmount'].sum().reindex(order)
    fig = go.Figure(go.Bar(x=age_rev.index, y=age_rev.values, marker_color='#6B7FA8'))
    fig.update_layout(title='Revenue by Age Group', font_family=FONT, height=350,
                       yaxis_title='Revenue ($)', plot_bgcolor='white', margin=dict(t=60, b=20), showlegend=False)
    return fig

age_chart(df).show()

## 8. Top customers by revenue

In [9]:
def top_customers(data, n=10):
    def mode_or_first(s):
        m = s.mode()
        return m.iloc[0] if not m.empty else s.iloc[0]

    agg = (data.groupby('CustomerID')
                .agg(Revenue=('PurchaseAmount', 'sum'),
                     Transactions=('PurchaseAmount', 'count'),
                     Segment=('Segment', mode_or_first),
                     Region=('CustomerRegion', mode_or_first),
                     Channel=('RetailChannel', mode_or_first),
                     AvgSatisfaction=('CustomerSatisfaction', 'mean'))
                .sort_values('Revenue', ascending=False)
                .head(n)
                .reset_index())
    agg['Revenue'] = agg['Revenue'].map(lambda v: f'${v:,.2f}')
    agg['AvgSatisfaction'] = agg['AvgSatisfaction'].round(1)
    return agg

top_customers(df)

,CustomerID,Revenue,Transactions,Segment,Region,Channel,AvgSatisfaction
0,12351,"$2,113.92",8,Growth,South,Online,3.90
1,12358,"$1,569.92",8,Growth,North,Online,4.10
2,12354,"$1,529.92",8,Growth,North,Physical Store,3.90
3,12356,"$1,525.92",8,Growth,East,Physical Store,4.20
4,12353,"$1,510.92",8,Growth,North,Online,3.90
5,12352,"$1,345.92",8,Growth,East,Physical Store,3.90
6,12350,"$1,329.92",8,Growth,West,Physical Store,4.40
7,12357,"$1,314.92",8,Growth,North,Online,3.80
8,12355,"$1,242.92",8,Growth,South,Online,4.10
9,12501,$549.99,1,Decline,East,Physical Store,1.00


## 9. Interactive explorer

Filter by segment, region, channel, category, age group, and gender — every chart and the KPI summary re-render together, just like the HTML dashboard. **Requires a live kernel** (widgets won't re-run in a static export like GitHub's preview).

In [10]:
seg_w = widgets.Dropdown(options=['All'] + SEGMENTS, value='All', description='Segment:')
region_w = widgets.Dropdown(options=['All'] + sorted(df['CustomerRegion'].unique().tolist()), value='All', description='Region:')
channel_w = widgets.Dropdown(options=['All'] + sorted(df['RetailChannel'].unique().tolist()), value='All', description='Channel:')
category_w = widgets.Dropdown(options=['All'] + sorted(df['Category'].unique().tolist()), value='All', description='Category:')
age_w = widgets.Dropdown(options=['All'] + [a for a in AGE_ORDER if a in df['CustomerAgeGroup'].unique()], value='All', description='Age:')
gender_w = widgets.Dropdown(options=['All'] + sorted(df['CustomerGender'].unique().tolist()), value='All', description='Gender:')

out = widgets.Output()

def render(*_):
    filtered = df.copy()
    if seg_w.value != 'All':
        filtered = filtered[filtered['Segment'] == seg_w.value]
    if region_w.value != 'All':
        filtered = filtered[filtered['CustomerRegion'] == region_w.value]
    if channel_w.value != 'All':
        filtered = filtered[filtered['RetailChannel'] == channel_w.value]
    if category_w.value != 'All':
        filtered = filtered[filtered['Category'] == category_w.value]
    if age_w.value != 'All':
        filtered = filtered[filtered['CustomerAgeGroup'] == age_w.value]
    if gender_w.value != 'All':
        filtered = filtered[filtered['CustomerGender'] == gender_w.value]

    with out:
        clear_output(wait=True)
        display(Markdown(f"**{len(filtered)} of {len(df)} records match the current filters**"))
        display(kpi_summary(filtered))
        if len(filtered):
            lifecycle_chart(filtered).show()
            trend_chart(filtered).show()
            region_category_channel(filtered).show()
            satisfaction_chart(filtered).show()
            age_chart(filtered).show()
            display(top_customers(filtered))
        else:
            display(Markdown("_No records match the current filter selection._"))

for w in (seg_w, region_w, channel_w, category_w, age_w, gender_w):
    w.observe(render, names='value')

filter_bar = widgets.HBox([seg_w, region_w, channel_w, category_w, age_w, gender_w])
display(filter_bar, out)
render()

Output()